# Kaggle Inference — Qwen2.5-VL-3B-Instruct (Data Pipeline Incident Response)

Runs the schema-drift agent using **Qwen/Qwen2.5-VL-3B-Instruct** (this is the HuggingFace repo name for the Qwen3-VL 4B model).

**VRAM budget on Kaggle T4 (15 GB):**
- Model weights in 4-bit NF4: ~2.5 GB
- Vision encoder (resident even for text tasks): ~0.8 GB
- KV cache + activations (max_new_tokens=512, history=16 turns): ~1.5 GB
- Framework overhead: ~0.4 GB
- **Total: ~5.2 GB** — leaves 9+ GB headroom, very safe.

---

## Kaggle Secrets required

Add these in **Settings → Secrets** before running:

| Secret name | What it is | Minimum permission |
|---|---|---|
| `GITHUB_TOKEN` | Classic Personal Access Token from github.com/settings/tokens | `repo` scope (needed to clone a private repo). If the repo is public, only `public_repo` is enough. |
| `HF_TOKEN` | HuggingFace token from huggingface.co/settings/tokens | **Read** access. If `Qwen/Qwen2.5-VL-3B-Instruct` is gated, you also need to accept the model license on the HF model page once while logged in — the token itself stays read-only. |

> **HF_TOKEN write access is NOT needed for inference.** Only needed later if you push a fine-tuned model to the Hub (training notebook).


In [9]:
import os, sys
from kaggle_secrets import UserSecretsClient

_s           = UserSecretsClient()
GITHUB_TOKEN = _s.get_secret('GITHUB_TOKEN')
HF_TOKEN     = _s.get_secret('HF_TOKEN')

# Hugging Face auth
os.environ['HF_TOKEN']               = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/standing-on-giants/Meta_hackathon.git'
REPO_DIR = '/kaggle/working/Meta_hackathon'
BRANCH   = 'dev/pratham'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git fetch --all --quiet')

# Checkout branch safely
os.system(f'cd {REPO_DIR} && git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}')
os.system(f'cd {REPO_DIR} && git pull origin {BRANCH} --quiet')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

print('Repo ready:', os.getcwd())
print('Current branch:', BRANCH)

Already on 'dev/pratham'


M	src/__pycache__/__init__.cpython-312.pyc
M	src/__pycache__/assertions.cpython-312.pyc
M	src/__pycache__/environment.cpython-312.pyc
M	src/__pycache__/models.cpython-312.pyc
M	src/__pycache__/pipeline_runner.cpython-312.pyc
M	src/__pycache__/tasks.cpython-312.pyc
Your branch is up to date with 'origin/dev/pratham'.
Repo ready: /kaggle/working/Meta_hackathon
Current branch: dev/pratham


In [10]:
# Install deps — qwen-vl-utils handles the Qwen2.5-VL image/video preprocessing pipeline
# We only do text inference here but the package is required by the model class
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate', 'bitsandbytes',
    'qwen-vl-utils', 'pandas', 'numpy', 'openai', 'python-dotenv'], check=True)

import bitsandbytes as bnb, torch
print(f'bitsandbytes : {bnb.__version__}')
print(f'torch        : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


bitsandbytes : 0.49.2
torch        : 2.10.0+cu128
CUDA         : True
GPU          : Tesla T4
VRAM         : 15.6 GB


In [11]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Qwen/Qwen2.5-VL-3B-Instruct is the HuggingFace model ID for the Qwen3-VL 4B class model.
# The product is called 'Qwen3-VL'; HF uses the Qwen2.5-VL series naming for the repo.
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# ── 4-bit NF4 quantization ─────────────────────────────────────────────
# T4 does NOT support bfloat16 natively — use float16 for compute dtype.
# double_quant: quantizes the quantization constants too (~0.1% quality, ~0 VRAM cost).
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── Processor (VL models use processor, not just tokenizer) ────────────
# min_pixels / max_pixels caps the resolution the vision encoder processes.
# For text-only inference this prevents accidental large image buffer allocation.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

# ── Model load ──────────────────────────────────────────────────────────
# device_map='auto' places layers on GPU first, spills to CPU if needed.
# With 4-bit the full 3B model is ~2.5 GB on GPU — leaves 12+ GB for KV cache.
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
model.eval()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded : {MODEL_ID}')
print(f'VRAM alloc   : {allocated:.2f} GB')
print(f'VRAM reserved: {reserved:.2f} GB')
print(f'VRAM free    : {total - reserved:.2f} GB  (of {total:.1f} GB)')


You are using a model of type qwen2_5_vl to instantiate a model of type qwen2_vl. This is not supported for all configurations of models and can yield errors.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/632 [00:00<?, ?it/s]

Qwen2VLForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-VL-3B-Instruct
Key                                               | Status     | 
--------------------------------------------------+------------+-
model.visual.blocks.{0...31}.mlp.gate_proj.weight | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.gate_proj.bias   | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.down_proj.bias   | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.down_proj.weight | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.up_proj.bias     | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.up_proj.weight   | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.fc1.bias         | MISSING    | 
model.visual.blocks.{0...31}.norm1.bias           | MISSING    | 
model.visual.blocks.{0...31}.mlp.fc1.weight       | MISSING    | 
model.visual.blocks.{0...31}.mlp.fc2.weight       | MISSING    | 
model.visual.blocks.{0...31}.norm2.bias           | MISSING    | 
model.visual.blocks.{0...31}.mlp.fc2.bias         | MISSING    |

Model loaded : Qwen/Qwen2.5-VL-3B-Instruct
VRAM alloc   : 3.53 GB
VRAM reserved: 3.70 GB
VRAM free    : 11.94 GB  (of 15.6 GB)


In [12]:
import re, json, textwrap
from typing import Any, Dict, List, Optional
from src.models import PipelineAction, PipelineObservation

MODEL_NAME  = MODEL_ID
MAX_TOKENS  = 512    # JSON actions are short; 512 is generous and keeps KV cache small
TEMPERATURE = 0.1
MAX_STEPS   = 25
BENCHMARK   = 'data_pipeline_incident_response'

SUCCESS_SCORE_THRESHOLD = 0.1

FALLBACK_ACTION = PipelineAction(action_type='compare_schema', params={'table': 'insights_ads'})


def _strip_think(text: str) -> str:
    """Remove <think>...</think> blocks.

    Qwen3 models emit chain-of-thought inside <think> tags before their answer.
    These are useful for debugging but break JSON parsing — strip them before
    trying to extract the action object.
    """
    return re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()


def generate(messages: list) -> str:
    """One inference pass through Qwen2.5-VL for text-only chat.

    Important differences vs LLaMA:
      - Must use processor.apply_chat_template, not tokenizer.apply_chat_template
      - Must call processor(text=...) to get input tensors (handles the VL preprocessing)
      - Must slice output_ids by input_len to get only new tokens
      - Must strip <think> tags from output before returning
    """
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], padding=True, return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=max(TEMPERATURE, 0.01),
            do_sample=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    raw = processor.tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True)
    return _strip_think(raw)


# ------------------------------------------------------------------ #
# Action parser — ported from inference_gemini_round2_schema_drift.py
# ------------------------------------------------------------------ #

def parse_llm_response(text: str) -> PipelineAction:
    """Extract, normalize and validate a PipelineAction from the model response."""
    payload = _extract_action_payload(text)
    if not payload:
        return FALLBACK_ACTION
    normalized = _normalize_action_payload(payload)
    try:
        return PipelineAction(**normalized)
    except Exception:
        return FALLBACK_ACTION


def _extract_action_payload(text: str) -> Optional[dict]:
    """Extract raw action JSON payload, including repair attempts for truncated output."""
    if not text:
        return None
    text = text.strip()
    if '```' in text:
        lines = text.split('\n')
        text = '\n'.join(l for l in lines if not l.strip().startswith('```'))
    start = text.find('{')
    if start == -1:
        return None
    end = text.rfind('}') + 1
    if end > start:
        try:
            data = json.loads(text[start:end])
            if isinstance(data, dict) and 'action_type' in data:
                return data
        except Exception:
            pass
    fragment = text[start:]
    repaired = _try_repair_json(fragment)
    if repaired and isinstance(repaired, dict):
        return repaired
    return None


def _normalize_action_payload(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Normalize action payload while preserving native handle_drift support."""
    action_type = str(payload.get('action_type', '')).strip()
    params = payload.get('params') or {}
    if not isinstance(params, dict):
        params = {}
    if action_type != 'handle_drift':
        return {'action_type': action_type, 'params': params}
    return {'action_type': 'handle_drift', 'params': params}


def _try_repair_json(fragment: str) -> Optional[dict]:
    """Try to repair truncated JSON from LLM output."""
    for suffix in ['"}}}', '"}}}', '"}}', '}}', '}']:
        try:
            data = json.loads(fragment + suffix)
            if isinstance(data, dict) and 'action_type' in data:
                return data
        except Exception:
            continue
    at_match = re.search(r'"\"action_type\"\\s*:\\s*\"([^\"]+)\"', fragment)
    if not at_match:
        at_match = re.search(r'"action_type"\s*:\s*"([^"]+)"', fragment)
    if not at_match:
        return None
    action_type = at_match.group(1)
    params: Dict[str, Any] = {}
    step_match   = re.search(r'"step_id"\s*:\s*"([^"]+)"', fragment)
    patch_match  = re.search(r'"patch_type"\s*:\s*"([^"]+)"', fragment)
    col_match    = re.search(r'"column"\s*:\s*"([^"]+)"', fragment)
    table_match  = re.search(r'"table"\s*:\s*"([^"]+)"', fragment)
    filter_match = re.search(r'"filter_condition"\s*:\s*"([^"]+)"', fragment)
    team_match   = re.search(r'"team"\s*:\s*"([^"]+)"', fragment)
    issue_match  = re.search(r'"issue_description"\s*:\s*"([^"]+)"', fragment)
    n_rows_match = re.search(r'"n_rows"\s*:\s*(\d+)', fragment)
    if step_match:   params['step_id']           = step_match.group(1)
    if patch_match:  params['patch_type']         = patch_match.group(1)
    if col_match:    params['column']             = col_match.group(1)
    if table_match:  params['table']              = table_match.group(1)
    if filter_match: params['filter_condition']   = filter_match.group(1)
    if team_match:   params['team']               = team_match.group(1)
    if issue_match:  params['issue_description']  = issue_match.group(1)
    if n_rows_match: params['n_rows']             = int(n_rows_match.group(1))
    try:
        return {'action_type': action_type, 'params': params}
    except Exception:
        return None


print('Helpers ready.')


Helpers ready.


In [13]:
SYSTEM_PROMPT = textwrap.dedent("""
You are an expert data engineer diagnosing and fixing broken data pipelines.

You will receive the current state of a pipeline (failing assertions, DAG structure,
historical run info) and must choose ONE action to take each turn.

WORKFLOW (follow this order strictly):
1. FIRST: read_data_sample on the raw input table(s) to see what the data looks like.
2. THEN: Use check_schema or compare_schema if a type or schema issue is suspected.
3. If you see any schema drift signal (renamed/missing columns, changed types, auth format drift,
   or stricter rate-limit behavior), use handle_drift.
4. THEN: Apply the RIGHT fix using add_data_filter or patch_transformation.
5. THEN: Call run_pipeline to verify the fix.
6. ONLY AFTER fixing what you can: If some data is genuinely corrupted (e.g. "N/A" values
   that cannot be parsed), call alert_upstream_team.
7. If assertions are still failing after run_pipeline, investigate more and apply
   additional fixes. Do NOT just call run_pipeline again without changing something.

AVAILABLE ACTIONS (respond with ONLY a JSON object, no markdown, no prose):

{"action_type": "read_data_sample", "params": {"table": "<table_name>", "n_rows": 20}}
{"action_type": "check_schema", "params": {"table": "<table_name>"}}
{"action_type": "compare_schema", "params": {"table": "<table_name>"}}
{"action_type": "handle_drift", "params": {"strategy": "<detect|numeric_format|null_fill|type_cast|join_key_prefix|filter_invalid|resolve_column_rename|alert_upstream>", "table": "<table_name_optional>", "step_id": "<step_id_optional>", "column": "<column_optional>", "old_column": "<optional_old_name>", "new_column": "<optional_new_name>", "filter_condition": "<optional>", "team": "<optional>", "issue_description": "<optional>"}}
{"action_type": "run_quality_assertion", "params": {"assertion_id": "<e.g. A1>"}}
{"action_type": "add_data_filter", "params": {"step_id": "<step_id>", "filter_condition": "<e.g. user_id IS NOT NULL>"}}
{"action_type": "patch_transformation", "params": {"step_id": "<step_id>", "patch_type": "<cast_column|coalesce|dedup|parse_currency|strip_prefix>", "column": "<column_name>"}}
{"action_type": "backfill_partition", "params": {"date": "<YYYY-MM-DD>"}}
{"action_type": "alert_upstream_team", "params": {"team": "<team_name_snake_case>", "issue_description": "<short description>"}}
{"action_type": "mark_acceptable", "params": {"assertion_id": "<id>", "reason": "<reason>"}}
{"action_type": "run_pipeline", "params": {}}

KEY PATCH TYPES (you can chain multiple patches on the same step — they run in order):
- parse_currency: Use when a column has mixed formats like "$1,234.56" and "1234.56" and "N/A".
  It strips $, commas, casts to float, and converts N/A to NaN. Works on ANY column with "N/A" strings,
  not just currency — e.g. if a numeric column like impressions has "N/A" values, use parse_currency on it.
- coalesce: Use AFTER parse_currency to replace NaN/null with a default value (default is 0).
  IMPORTANT: After parse_currency, NaN values will cause value_range assertions to fail.
  You MUST chain a coalesce patch to fix this: {"action_type": "patch_transformation", "params": {"step_id": "<same_step>", "patch_type": "coalesce", "column": "<same_column>"}}
  If coalescing a denominator column (e.g. impressions used in CTR = clicks/impressions), coalescing to 0
  will cause division by zero. Instead, filter out those rows: add_data_filter with "column IS NOT NULL".
- cast_column: Use when a column needs simple numeric casting.
- dedup: Use when there are duplicate rows based on a key column.
  IMPORTANT: If a "unique" assertion is failing, the fix is ALWAYS dedup on the failing column.
  Do NOT use coalesce or add_data_filter for uniqueness failures — only dedup works.
- strip_prefix: Use when column values have a spurious prefix like "CMP_" that needs removal.
  Params: step_id, column. Optionally "prefix" (default "CMP_"). After stripping, chain cast_column
  if the underlying value should be numeric.

DRIFT HANDLING RULES:
- Use handle_drift when schema or contract changes between runs.
- handle_drift strategy mapping:
    detect -> compare_schema
    numeric_format -> patch_transformation(parse_currency)
    null_fill -> patch_transformation(coalesce)
    type_cast -> patch_transformation(cast_column)
    join_key_prefix -> patch_transformation(strip_prefix)
    filter_invalid -> add_data_filter
    resolve_column_rename -> restore compatibility for renamed columns (e.g. spend <- total_spend)
    alert_upstream -> alert_upstream_team
- For spend -> total_spend style drift, compare schema first, then patch transformations to align types.

UPSTREAM TEAM NAMING:
- Team names are always lowercase snake_case. Examples: meta_ads_api_team, data_engineering, vendor_support.
- If the description mentions "Meta", "Graph API", or "Meta Ads", the team is likely "meta_ads_api_team".

RULES:
- RESPOND WITH ONLY A JSON OBJECT. No markdown fences, no explanation, no prose.
- Do NOT call run_pipeline unless you applied a filter or patch since the last run.
- Do NOT apply a fix before reading the data — this will be penalised.
- NEVER use mark_acceptable. It always results in a heavy penalty. Instead, fix the data.
- After parse_currency, ALWAYS chain coalesce on the same column to handle NaN values before calling run_pipeline.
- If a "unique" assertion fails (e.g. uniqueness on order_item_id), the ONLY correct fix is dedup.
  Do NOT try coalesce, add_data_filter, or any other patch for uniqueness failures.
- If a computed column (like CTR) has a value_range failure, check ALL input columns in its formula.
  For example, if CTR = clicks/impressions and impressions has "N/A" strings, you must fix impressions
  with parse_currency first, then filter out null rows, before the computed column can produce valid values.
- If a joined output table has 0 rows (row_count assertion failing), the join keys likely don't match.
  Use compare_schema on the input tables to detect type/format drifts like string vs int, or unwanted
  prefixes on the join key. Apply strip_prefix + cast_column to align the keys.
- If pipeline_passed is true, you are done — unless the task description mentions alerting an upstream team.
- NEVER repeat the same action you already tried. If an action did not fix the problem, try a DIFFERENT action.
""").strip()


def _collect_schema_drift_signals(obs: PipelineObservation) -> List[str]:
    signals: List[str] = []
    desc = (obs.description or "").lower()
    if "schema drift" in desc or "contract" in desc:
        signals.append("Task description references schema/contract drift.")
    if obs.schema_diff:
        schema_diff_json = json.dumps(obs.schema_diff).lower()
        if "removed" in schema_diff_json:
            signals.append("Historical columns appear removed in current schema.")
        if "changed" in schema_diff_json:
            signals.append("Column types differ from historical schema.")
        if "new" in schema_diff_json:
            signals.append("New columns detected relative to historical schema.")
    for r in obs.failed_assertions:
        actual = (r.actual or "").lower()
        if "missing" in actual and "column" in actual:
            signals.append(f"Assertion {r.assertion_id} reports a missing column.")
        if "not found" in actual and "column" in actual:
            signals.append(f"Assertion {r.assertion_id} reports a renamed or deleted column.")
        if "type" in actual and ("object" in actual or "string" in actual):
            signals.append(f"Assertion {r.assertion_id} indicates possible type drift.")
    deduped: List[str] = []
    for s in signals:
        if s not in deduped:
            deduped.append(s)
    return deduped[:6]


def _detect_action_loop(actions_taken: List[str]) -> Optional[str]:
    # Detect if the last 3+ actions share the same action_type(params) pattern.
    # actions_taken format: "[step] action_type({'param': 'val'})"
    if len(actions_taken) < 3:
        return None

    def _extract_action_key(action_str: str) -> str:
        # Extract "action_type({params})" from "[N] action_type({params})"
        idx = action_str.find(']')
        if idx >= 0:
            return action_str[idx+1:].strip()
        return action_str.strip()

    last_3 = [_extract_action_key(a) for a in actions_taken[-3:]]
    if last_3[0] == last_3[1] == last_3[2]:
        return last_3[0]

    # Also detect 2-step oscillation: A, B, A, B
    if len(actions_taken) >= 4:
        last_4 = [_extract_action_key(a) for a in actions_taken[-4:]]
        if last_4[0] == last_4[2] and last_4[1] == last_4[3]:
            return f"{last_4[0]} and {last_4[1]} (oscillating)"

    return None


def _build_loop_hint(obs: PipelineObservation, looped_action: str) -> str:
    # Build a targeted hint based on what the model is stuck on and what assertions are failing
    hint = (
        f"\n[CRITICAL LOOP DETECTED]: You have been repeating '{looped_action}' "
        f"without making progress. This action is NOT solving the problem. "
        f"You MUST try a COMPLETELY DIFFERENT action type.\n"
    )

    # Analyze the failing assertions to suggest the right fix
    for r in obs.failed_assertions:
        atype = (r.assertion_type or "").lower()
        col = r.column or ""
        table = r.table or ""

        if atype == "unique":
            hint += (
                f"\n  -> Assertion {r.assertion_id} is a UNIQUENESS failure on column '{col}' "
                f"in table '{table}'. The ONLY fix for this is: "
                f'{{"action_type": "patch_transformation", "params": {{"step_id": "<find the step that outputs {table}>", "patch_type": "dedup", "column": "{col}"}}}}'
            )
        elif atype == "row_count":
            hint += (
                f"\n  -> Assertion {r.assertion_id} is a ROW COUNT failure on '{table}'. "
                f"This is usually caused by duplicate rows inflating the count. "
                f"Look for a uniqueness assertion on the same table and fix that with dedup first."
            )
        elif atype == "type_check":
            hint += (
                f"\n  -> Assertion {r.assertion_id} is a TYPE CHECK failure on '{col}' in '{table}'. "
                f"Use parse_currency on that column to convert string values to numeric."
            )
        elif atype == "value_range":
            hint += (
                f"\n  -> Assertion {r.assertion_id} is a VALUE RANGE failure on '{col}'. "
                f"Check if parse_currency was applied — if so, chain coalesce to replace NaN with 0."
            )

    # If the model is stuck on run_pipeline
    if "run_pipeline" in looped_action:
        hint += (
            "\n\n  You MUST apply a fix (patch_transformation or add_data_filter) "
            "BEFORE calling run_pipeline again. run_pipeline without changes does nothing."
        )

    return hint


def build_user_prompt(obs: PipelineObservation, step: int) -> str:
    failed_str = "\n".join(
        f"  [{r.assertion_id}] {r.assertion_type} on {r.table}"
        f"({r.column or 'N/A'}): {r.actual}"
        for r in obs.failed_assertions
    ) or "  (none -- all passing!)"

    passed_str = ", ".join(r.assertion_id for r in obs.passed_assertions) or "none"

    dag_str = "\n".join(
        f"  {n.step_id}: {n.input_table} -> {n.output_table}"
        + (f" | filters: {n.applied_filters}" if n.applied_filters else "")
        + (f" | patches: {n.applied_patches}" if n.applied_patches else "")
        for n in obs.dag_structure
    )

    hist_str = "\n".join(
        f"  {r.date}: {r.status} ({r.row_count} rows)"
        for r in obs.historical_runs
    )

    sample_str = ""
    if obs.data_sample:
        sample_rows = obs.data_sample[:5]
        null_rows = [r for r in obs.data_sample if any(v is None for v in r.values())]
        if null_rows:
            sample_str = (
                "\nDATA SAMPLE (first 5 rows):\n"
                + json.dumps(sample_rows, indent=2, default=str)
                + f"\nROWS WITH NULL VALUES ({len(null_rows)} found):\n"
                + json.dumps(null_rows[:5], indent=2, default=str)
            )
        else:
            sample_str = (
                "\nDATA SAMPLE (first 5 rows):\n"
                + json.dumps(sample_rows, indent=2, default=str)
            )

    schema_str = ""
    if obs.current_schema:
        schema_str = "\nCURRENT SCHEMA: " + json.dumps(obs.current_schema)
    if obs.schema_diff:
        schema_str += "\nSCHEMA DIFF vs HISTORICAL: " + json.dumps(obs.schema_diff)

    drift_signals = _collect_schema_drift_signals(obs)
    drift_str = ""
    if drift_signals:
        drift_str = "\nSCHEMA DRIFT SIGNALS:\n" + "\n".join(f"  - {s}" for s in drift_signals)

    actions_str = "\n".join(f"  {a}" for a in obs.actions_taken[-5:]) or "  (none yet)"

    read_actions  = sum(1 for a in obs.actions_taken if "read_data_sample" in a or "check_schema" in a)
    fix_actions   = sum(1 for a in obs.actions_taken if "add_data_filter" in a or "patch_transformation" in a)
    mark_actions  = sum(1 for a in obs.actions_taken if "mark_acceptable" in a)
    parse_done    = any("parse_currency" in a for a in obs.actions_taken)
    coalesce_done = any("coalesce" in a for a in obs.actions_taken)

    hint_str = ""

    # -- Loop detection (prompt-based, not harness override) --
    looped_action = _detect_action_loop(obs.actions_taken)
    if looped_action:
        hint_str += _build_loop_hint(obs, looped_action)
    else:
        # Standard hints (only if not already showing loop hint)
        if read_actions >= 2 and fix_actions == 0:
            hint_str = (
                "\n[HINT]: You have already read the data. "
                "Stop diagnosing. Apply a fix now using add_data_filter or patch_transformation, "
                "then call run_pipeline."
            )

    value_range_failing = any(
        r.assertion_type == "value_range" and "non-numeric" in r.actual
        for r in obs.failed_assertions
    )
    if parse_done and not coalesce_done and value_range_failing:
        hint_str += (
            "\n[CRITICAL]: A value_range assertion is STILL failing because parse_currency converts "
            "unparseable values (like 'N/A') to NaN, and NaN counts as out-of-range. "
            "You MUST apply a coalesce patch to replace NaN with 0 on the same column and step "
            "where you applied parse_currency."
        )
    if mark_actions >= 1:
        hint_str += (
            "\n[WARNING]: NEVER use mark_acceptable again. It gives a -1.0 penalty every time. "
            "Instead, apply a coalesce patch to fix NaN values, then run_pipeline."
        )
    recent = obs.actions_taken[-3:]
    recent_runs = sum(1 for a in recent if "run_pipeline" in a)
    if recent_runs >= 2 and not obs.pipeline_passed:
        hint_str += (
            "\n[CRITICAL]: You have called run_pipeline multiple times with no progress. "
            "You MUST apply a fix (patch_transformation or add_data_filter) before calling run_pipeline again."
        )

    return textwrap.dedent(f"""
    STEP {step}/{obs.max_steps}
    TASK: {obs.task_id} ({obs.difficulty})
    DESCRIPTION: {obs.description}
    PIPELINE PASSED: {obs.pipeline_passed}
    LAST ACTION RESULT: {obs.last_action_result}

    DAG STRUCTURE:
    {dag_str}

    FAILING ASSERTIONS:
    {failed_str}

    PASSING ASSERTIONS: {passed_str}

    HISTORICAL RUNS:
    {hist_str}

    RECENT ACTIONS TAKEN:
    {actions_str}
    {sample_str}{schema_str}{drift_str}{hint_str}

    Respond with exactly ONE action JSON object.
    """).strip()


print('Prompts ready.')


Prompts ready.


In [14]:
from src.environment import DataPipelineEnv


def log_start(task: str, env: str, model: str) -> None:
    print(f'[START] task={task} env={env} model={model}', flush=True)

def log_step(step: int, action: str, reward: float, done: bool, error: Optional[str]) -> None:
    error_val   = error if error else "null"
    done_val    = str(done).lower()
    action_safe = action.replace("\n", " ").replace("\r", "")[:200]
    print(
        f"[STEP] step={step} action={action_safe} reward={reward:.2f} "
        f"done={done_val} error={error_val}",
        flush=True,
    )

def log_end(success: bool, steps: int, score: float, rewards: List[float]) -> None:
    rewards_str = ",".join(f"{r:.2f}" for r in rewards)
    print(
        f"[END] success={str(success).lower()} steps={steps} "
        f"score={score:.2f} rewards={rewards_str}",
        flush=True,
    )


def run_episode(task_id: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> dict:
    env = DataPipelineEnv(task_id=task_id)

    history:         List[dict] = []
    rewards:         List[float] = []
    steps_taken:     int = 0
    score:           float = 0.0
    success:         bool = False
    n_passed:        int = 0
    n_total:         int = 0
    pipeline_passed: bool = False

    log_start(task=task_id, env=BENCHMARK, model=MODEL_NAME)

    try:
        obs = env.reset()
        if verbose:
            print(f'\n{"="*60}', file=sys.stderr)
            print(f'TASK: {task_id.upper()}  |  {len(obs.failed_assertions)} assertions failing', file=sys.stderr)
            print(f'{"="*60}', file=sys.stderr)
            print(f'Description: {obs.description}', file=sys.stderr)

        for step in range(1, max_steps + 1):
            if obs.pipeline_passed:
                if verbose:
                    print(f'\n[PASSED] Pipeline passed at step {step - 1}!', file=sys.stderr)
                break

            # If the model is in a loop, trim history aggressively to break the pattern
            looped = _detect_action_loop(obs.actions_taken)
            if looped:
                if verbose:
                    print(f'  [Step {step}] Loop detected: {looped}. Trimming history.', file=sys.stderr)
                # Keep only last 2 turns so the model loses the repetitive context
                history = history[-2:]

            user_prompt = build_user_prompt(obs, step)
            history.append({'role': 'user', 'content': user_prompt})
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history

            torch.cuda.empty_cache()

            response_text = ''
            try:
                response_text = generate(messages)
            except torch.cuda.OutOfMemoryError:
                print(f'[OOM] Step {step}: trimming history to 4 turns and retrying.', file=sys.stderr)
                history = history[-4:]
                messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history
                torch.cuda.empty_cache()
                try:
                    response_text = generate(messages)
                except Exception as e2:
                    print(f'[OOM-RETRY-FAIL] {e2}', file=sys.stderr)
            except Exception as exc:
                if verbose:
                    print(f'  [Step {step}] Generation error: {exc}. Using fallback.', file=sys.stderr, flush=True)

            action = parse_llm_response(response_text)

            # Smart fallback: empty response -> diagnostic instead of run_pipeline
            if action.action_type == 'run_pipeline' and not response_text.strip():
                if obs.failed_assertions:
                    target_table = obs.failed_assertions[0].table
                    action = PipelineAction(
                        action_type='compare_schema',
                        params={'table': target_table}
                    )

            history.append({'role': 'assistant', 'content': response_text or '{}'})
            if len(history) > 14:
                history = history[-14:]

            result = env.step(action)
            obs    = result.observation
            reward = result.reward or 0.0
            done   = result.done
            error: Optional[str] = None

            rewards.append(reward)
            steps_taken = step

            log_step(
                step=step,
                action=json.dumps(action.model_dump()).replace("\n", " ")[:200],
                reward=reward,
                done=done,
                error=error,
            )

            if verbose:
                print(f'\n[Step {step}] Raw response: {response_text[:120]}', file=sys.stderr)
                print(f'[Step {step}] Action: {action.action_type}({action.params})', file=sys.stderr)
                print(
                    f'  Reward: {reward:+.2f} | '
                    f'Passed: {len(obs.passed_assertions)}/{len(obs.failed_assertions)+len(obs.passed_assertions)} | '
                    f'Result: {obs.last_action_result[:80]}',
                    file=sys.stderr
                )

            if done:
                break

        n_total  = len(obs.failed_assertions) + len(obs.passed_assertions)
        n_passed = len(obs.passed_assertions)
        pipeline_passed = obs.pipeline_passed
        raw_score = n_passed / n_total if n_total > 0 else 0.0
        score   = min(max(raw_score, 0.01), 0.99)
        success = score >= SUCCESS_SCORE_THRESHOLD

        if verbose:
            print(f'\n--- Episode Summary ---', file=sys.stderr)
            print(f'  Score (assertion pass rate): {score:.2f}', file=sys.stderr)
            print(f'  Total reward:                {sum(rewards):.2f}', file=sys.stderr)
            print(f'  Steps taken:                 {steps_taken}', file=sys.stderr)
            print(f'  Pipeline passed:             {pipeline_passed}', file=sys.stderr)

    except Exception as exc:
        import traceback
        print(f'[ERROR] {task_id}: {exc}', file=sys.stderr)
        traceback.print_exc(file=sys.stderr)
    finally:
        try:
            env.close()
        except AttributeError:
            pass
        except Exception as e:
            print(f'[DEBUG] env.close() error: {e}', file=sys.stderr, flush=True)
        log_end(success=success, steps=steps_taken, score=score, rewards=rewards)

    return {
        'task_id':           task_id,
        'score':             round(score, 4),
        'success':           success,
        'pipeline_passed':  pipeline_passed,
        'total_reward':     round(sum(rewards), 4),
        'steps_taken':      steps_taken,
        'assertions_passed': n_passed,
        'assertions_total':  n_total,
    }


print('Runner ready.')


Runner ready.


In [15]:
# Dynamically detect which tasks are available in the cloned branch
from src.tasks import TASKS as _AVAILABLE_TASKS

ALL_TASKS = ['easy', 'medium', 'hard', 'hard2']
VALID_TASKS = [t for t in ALL_TASKS if t in _AVAILABLE_TASKS]

print(f'Available tasks in this branch: {VALID_TASKS}', file=sys.stderr)

# Change TASKS_TO_RUN to run a single task for faster testing
# Options: 'easy' | 'medium' | 'hard' | 'hard2' | 'all'
TASKS_TO_RUN = 'all'

task_ids = VALID_TASKS if TASKS_TO_RUN == 'all' else [TASKS_TO_RUN]

all_results = []
for task_id in task_ids:
    if task_id not in _AVAILABLE_TASKS:
        print(f'[SKIP] Task "{task_id}" not available in this branch.', file=sys.stderr)
        continue
    result = run_episode(task_id=task_id, max_steps=MAX_STEPS, verbose=True)
    all_results.append(result)
    torch.cuda.empty_cache()   # release KV cache between tasks

print('\n' + '='*60, file=sys.stderr)
print('FINAL SCORES', file=sys.stderr)
print('='*60, file=sys.stderr)
total = 0.0
for r in all_results:
    status = '[PASSED]' if r['pipeline_passed'] else '[FAILED]'
    print(f"  {r['task_id']:8s} | score={r['score']:.2f} | "
          f"reward={r['total_reward']:+.2f} | steps={r['steps_taken']:2d} | {status}", file=sys.stderr)
    total += r['score']
avg = total / len(all_results) if all_results else 0.0
print(f'\n  Avg score: {avg:.4f}', file=sys.stderr)
print('\nJSON_RESULTS:', json.dumps(all_results, indent=2), file=sys.stderr)


[START] task=easy env=data_pipeline_incident_response model=Qwen/Qwen2.5-VL-3B-Instruct


Available tasks in this branch: ['easy', 'medium', 'hard', 'hard2']

TASK: EASY  |  1 assertions failing
Description: Your nightly orders pipeline failed. The orders_clean table has a NOT NULL assertion on user_id that is now failing (5 of 100 rows). Investigate raw_orders, identify the root cause, and fix the pipeline.


[STEP] step=1 action={"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 20}} reward=-0.05 done=false error=null



[Step 1] Raw response: {"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 20}}
[Step 1] Action: read_data_sample({'table': 'raw_orders', 'n_rows': 20})
  Reward: -0.05 | Passed: 2/3 | Result: Showing 20 rows from 'raw_orders' (unfiltered).


[STEP] step=2 action={"action_type": "check_schema", "params": {"table": "raw_orders"}} reward=0.00 done=false error=null


...


[Step 18] Raw response: {
  "action_type": "run_pipeline",
  "params": {}
}
[Step 18] Action: run_pipeline({})
  Reward: +0.00 | Passed: 2/8 | Result: Pipeline re-run: 2/8 assertions passing. +0 gained, -0 lost.
  [Step 19] Loop detected: run_pipeline({}). Trimming history.


[STEP] step=19 action={"action_type": "patch_transformation", "params": {"step_id": "transform_insights", "patch_type": "parse_currency", "column": "spend"}} reward=-0.50 done=false error=null



[Step 19] Raw response: {
  "action_type": "patch_transformation",
  "params": {
    "step_id": "transform_insights",
    "patch_type": "parse_c
[Step 19] Action: patch_transformation({'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'spend'})
  Reward: -0.50 | Passed: 2/8 | Result: Patch 'parse_currency' on column 'spend' applied to step 'transform_insights'. [


[STEP] step=20 action={"action_type": "run_pipeline", "params": {}} reward=0.00 done=true error=null
[END] success=true steps=20 score=0.25 rewards=-0.05,-0.10,-0.50,0.00,0.00,-0.50,0.00,0.00,0.00,-0.50,-0.50,0.00,0.00,0.00,-0.50,0.00,0.00,0.00,-0.50,0.00



[Step 20] Raw response: {
  "action_type": "run_pipeline",
  "params": {}
}
[Step 20] Action: run_pipeline({})
  Reward: +0.00 | Passed: 2/8 | Result: Pipeline re-run: 2/8 assertions passing. +0 gained, -0 lost. [WARNING] Max steps

--- Episode Summary ---
  Score (assertion pass rate): 0.25
  Total reward:                -3.15
  Steps taken:                 20
  Pipeline passed:             False

FINAL SCORES
  easy     | score=0.99 | reward=+1.35 | steps= 4 | [PASSED]
  medium   | score=0.99 | reward=+1.75 | steps= 7 | [PASSED]
  hard     | score=0.25 | reward=-2.55 | steps=20 | [FAILED]
  hard2    | score=0.25 | reward=-3.15 | steps=20 | [FAILED]

  Avg score: 0.6200

JSON_RESULTS: [
  {
    "task_id": "easy",
    "score": 0.99,
    "success": true,
    "pipeline_passed": true,
    "total_reward": 1.35,
    "steps_taken": 4,
    "assertions_passed": 3,
    "assertions_total": 3
  },
  {
    "task_id": "medium",
    "score": 0.99,
    "success": true,
    "pipeline_passed": true,
 

In [16]:
# Run any time to check VRAM health
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total_vr  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM allocated : {allocated:.2f} GB')
print(f'VRAM reserved  : {reserved:.2f} GB')
print(f'VRAM free      : {total_vr - reserved:.2f} GB  (of {total_vr:.1f} GB)')


VRAM allocated : 3.53 GB
VRAM reserved  : 3.70 GB
VRAM free      : 11.94 GB  (of 15.6 GB)
